In [0]:
CATALOG = "olist_ecommerce"
BRONZE_SCHEMA = "bronze"
RAW_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw_data"

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")

In [0]:
df_orders = (
    spark.
    read.
    csv(f"{RAW_PATH}/olist_orders_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
)
# df_orders.count()

In [0]:
df_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist_ecommerce.bronze.orders")

In [0]:
%sql
SELECT *
FROM olist_ecommerce.bronze.orders
LIMIT 10;

In [0]:
df_order_items = spark.read.csv(f"{RAW_PATH}/olist_order_items_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_products = spark.read.csv(f"{RAW_PATH}/olist_products_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_customers = spark.read.csv(f"{RAW_PATH}/olist_customers_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_sellers = spark.read.csv(f"{RAW_PATH}/olist_sellers_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_payments = spark.read.csv(f"{RAW_PATH}/olist_order_payments_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_reviews = spark.read.csv(f"{RAW_PATH}/olist_order_reviews_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_category_translation = spark.read.csv(f"{RAW_PATH}/product_category_name_translation.csv", sep=',', header=True, inferSchema=True, encoding='latin1')
df_geolocation = spark.read.csv(f"{RAW_PATH}/olist_geolocation_dataset.csv", sep=',', header=True, inferSchema=True, encoding='latin1')

In [0]:
df_order_items.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.order_items")
df_products.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.products")
df_customers.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.customers")
df_sellers.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.sellers")
df_payments.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.payments")
df_reviews.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.reviews")
df_category_translation.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.category_translation")
df_geolocation.write.format("delta").mode("overwrite").saveAsTable("olist_ecommerce.bronze.geolocation")

In [0]:
print("Total:", df_orders.count())

print(
    df_orders.select("order_id").distinct().count()
)

In [0]:
from pyspark.sql.functions import to_timestamp

df_orders_clean = (
    df_orders
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    )
)
df_orders_clean.printSchema()